# Data Loading

This script is needed to load the experiment data into the Sqlite database used. You can customize the input reports by modifying the `experiment_name` and `experiment_data_path` variables (by default, they will read from the output folder)

In [16]:
import logging
import pandas as pd
from helpers.sqlite_helpers import pd_to_sqlDB, sql_query_to_pd, run_sql_query

experiment_name = "javascript"
experiment_data_path = f"../executions/{experiment_name}/output/reports"

TOOLS = ['diff3', 'last_merge', 'mergiraf']
LANGUAGES = ['javascript']

## Creation of initial tables

### Loading data from tool executions on scenarios

In [17]:
def get_df_for_language_tool(language: str, tool: str) -> pd.DataFrame:
    tool_executions_df = pd.read_csv(f'{experiment_data_path}/merge-tools/{tool}.csv', names=['project', 'merge_sha', 'file_path', 'output_file_path', 'result', 'time_in_ns'])
    tool_executions_df['scenario_id'] = tool_executions_df['project'] + ':' + tool_executions_df['merge_sha']
    tool_executions_df['language'] = language.upper()
    tool_executions_df['tool'] = tool.upper()
    return tool_executions_df

global_executions_df = pd.concat([get_df_for_language_tool(language, tool) for tool in TOOLS for language in LANGUAGES], ignore_index=True)
pd_to_sqlDB(global_executions_df, f"global_executions")
sql_query_to_pd("SELECT * FROM global_executions")

2026-07-20 17:42:42 INFO: SQL DB default.db created
2026-07-20 17:42:42 INFO: SQL Table global_executions created with 9 columns
2026-07-20 17:42:42 INFO: SQL Table global_executions created with 9 columns
2026-07-20 17:42:42 INFO: 7371 rows uploaded to global_executions


,project,merge_sha,file_path,output_file_path,result,time_in_ns,scenario_id,language,tool
0,trix,e3d72b393f03eae660b61fafce1e79cc8e37f2af,output/trix/e3d72b393f03eae660b61fafce1e79cc8e...,output/trix/e3d72b393f03eae660b61fafce1e79cc8e...,SUCCESS_WITHOUT_CONFLICTS,99935623,trix:e3d72b393f03eae660b61fafce1e79cc8e37f2af,JAVASCRIPT,DIFF3
1,eleventy,c0b482bbd7c63fb44f34d05fdbccc2fca77a92bf,output/eleventy/c0b482bbd7c63fb44f34d05fdbccc2...,output/eleventy/c0b482bbd7c63fb44f34d05fdbccc2...,SUCCESS_WITHOUT_CONFLICTS,14429339,eleventy:c0b482bbd7c63fb44f34d05fdbccc2fca77a92bf,JAVASCRIPT,DIFF3
2,eleventy,c0b482bbd7c63fb44f34d05fdbccc2fca77a92bf,output/eleventy/c0b482bbd7c63fb44f34d05fdbccc2...,output/eleventy/c0b482bbd7c63fb44f34d05fdbccc2...,SUCCESS_WITHOUT_CONFLICTS,18451903,eleventy:c0b482bbd7c63fb44f34d05fdbccc2fca77a92bf,JAVASCRIPT,DIFF3
3,Aloha-Editor,de19e8dea332e55e22391133127d64cb39efa6eb,output/Aloha-Editor/de19e8dea332e55e2239113312...,output/Aloha-Editor/de19e8dea332e55e2239113312...,SUCCESS_WITH_CONFLICTS,20518001,Aloha-Editor:de19e8dea332e55e22391133127d64cb3...,JAVASCRIPT,DIFF3
4,Aloha-Editor,de19e8dea332e55e22391133127d64cb39efa6eb,output/Aloha-Editor/de19e8dea332e55e2239113312...,output/Aloha-Editor/de19e8dea332e55e2239113312...,SUCCESS_WITH_CONFLICTS,37101991,Aloha-Editor:de19e8dea332e55e22391133127d64cb3...,JAVASCRIPT,DIFF3
...,...,...,...,...,...,...,...,...,...
7366,awesome-uses,4affd2bf6c2f478067f0583a41bde6ed4524c191,output/awesome-uses/4affd2bf6c2f478067f0583a41...,output/awesome-uses/4affd2bf6c2f478067f0583a41...,SUCCESS_WITHOUT_CONFLICTS,247849131,awesome-uses:4affd2bf6c2f478067f0583a41bde6ed4...,JAVASCRIPT,MERGIRAF
7367,awesome-uses,f8b873b0c6bccbf00fb891375ada3edfc226f220,output/awesome-uses/f8b873b0c6bccbf00fb891375a...,output/awesome-uses/f8b873b0c6bccbf00fb891375a...,SUCCESS_WITHOUT_CONFLICTS,410947759,awesome-uses:f8b873b0c6bccbf00fb891375ada3edfc...,JAVASCRIPT,MERGIRAF
7368,awesome-uses,887fbd26777ea37869ee98eab215f88dce03f143,output/awesome-uses/887fbd26777ea37869ee98eab2...,output/awesome-uses/887fbd26777ea37869ee98eab2...,SUCCESS_WITHOUT_CONFLICTS,424612034,awesome-uses:887fbd26777ea37869ee98eab215f88dc...,JAVASCRIPT,MERGIRAF
7369,awesome-uses,f99460932ed37978a5f0614eb448eb35de08febf,output/awesome-uses/f99460932ed37978a5f0614eb4...,output/awesome-uses/f99460932ed37978a5f0614eb4...,SUCCESS_WITHOUT_CONFLICTS,228884325,awesome-uses:f99460932ed37978a5f0614eb448eb35d...,JAVASCRIPT,MERGIRAF


## Aggregations

### Information about execution per scenery on each tool (excluding failures)

In [18]:
def get_executions_per_commit_query() -> str:
    return f"""
        WITH global_executions_per_commit AS (
            SELECT
                e.tool,
                e.tool,
                e.language,
                e.scenario_id,
                e.project,
                e.merge_sha,
                SUM(time_in_ns) as time_in_ns,
                CASE 
                    WHEN SUM(result = "TOOL_ERROR") > 0 THEN "TOOL_ERROR"
                    WHEN SUM(result = "SUCCESS_WITH_CONFLICTS") > 0 THEN "SUCCESS_WITH_CONFLICTS"
                    ELSE "SUCCESS_WITHOUT_CONFLICTS"
                END AS result
            FROM
                global_executions e
            GROUP BY
                e.scenario_id,
                e.tool,
                e.language
        )
        SELECT
            *
        FROM
            global_executions_per_commit gepc
        WHERE
            NOT EXISTS (SELECT 1 FROM global_executions_per_commit gepc1 WHERE gepc.scenario_id = gepc1.scenario_id AND gepc.result = "TOOL_ERROR")
            AND (SELECT COUNT(DISTINCT gepc1.tool) FROM global_executions_per_commit gepc1 WHERE gepc.scenario_id = gepc1.scenario_id) = {TOOLS.__len__()}
    """

logging.info(f"Creating view global_executions_per_commit")
run_sql_query(f"""DROP VIEW IF EXISTS global_executions_per_commit""")
run_sql_query(f"""CREATE VIEW global_executions_per_commit AS {get_executions_per_commit_query()}""")

2026-07-20 17:42:42 INFO: Creating view global_executions_per_commit


### Metadata

In [19]:
commits_df = pd.read_csv(f'{experiment_data_path}/commits.csv', names=['project', 'merge', 'right', 'left'])
commits_df['scenario_id'] = commits_df['project'] + ':' + commits_df['merge']
pd_to_sqlDB(commits_df, "commits")

2026-07-20 17:42:42 INFO: SQL DB default.db created
2026-07-20 17:42:42 INFO: SQL Table commits created with 5 columns
2026-07-20 17:42:42 INFO: SQL Table commits created with 5 columns
2026-07-20 17:42:42 INFO: 1449 rows uploaded to commits


In [20]:
projects_df = pd.read_csv(f'{experiment_data_path}/projects.csv', names=['path', 'name'])
pd_to_sqlDB(projects_df, "projects")

2026-07-20 17:42:42 INFO: SQL DB default.db created
2026-07-20 17:42:42 INFO: SQL Table projects created with 2 columns
2026-07-20 17:42:42 INFO: SQL Table projects created with 2 columns
2026-07-20 17:42:42 INFO: 92 rows uploaded to projects
